<a href="https://colab.research.google.com/github/Heng1222/MalSem-Decon/blob/main/malicious_concept_subspace_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Facebook 社團名稱惡意語意分析：Concept Subspace Pipeline

這份 notebook 實作一條可追蹤、可標註、可解釋的流程，用「概念子空間（Concept Subspace）」偵測 Facebook 社團名稱中的負面內容，並用語境風險權重處理一字多義問題。

流程涵蓋：

1. **Seed-based Corpus Filtering**：用 `negative_keywords.txt` 從 `nodes_page_1hop.csv` 篩出候選惡意文本，並用 `ckiplab/bert-base-chinese` 產生 embedding 與 UMAP 3D 視覺化。
2. **Embedding Space Clustering & Labeling**：K-Means 分成 30 群，抽出每群離中心最近的 5 筆，提供人工標註介面與自動初標。
3. **Contextual Co-occurrence & Attention MLP**：用 BERT contextual token embedding 訓練淺層 MLP，輸出每個 token 的 0~1 語境風險權重。
4. **Subspace Definition & Disambiguation**：用人工確認的惡意 cluster 建立概念子空間，並計算 token 對子空間的 projection norm / angle cosine。
5. **Pipeline Integration & Attention Rollback**：融合 MLP 語境風險與子空間投影，實作 `evaluate_group_name(text)`，回傳是否惡意、命中概念與貢獻詞。


In [1]:
# 若環境缺少套件，請先取消下一行註解後執行。
# 這個 cell 只負責環境、路徑與全域參數；真正流程從 Step 1 開始。
!pip3 install -q pandas numpy scikit-learn transformers torch umap-learn plotly ipywidgets tqdm joblib

from __future__ import annotations

# 標準函式庫：處理 HTML 視覺化、JSON/CSV 序列化、路徑與文字規則。
import html
import json
import pickle
import random
import re
import warnings
from collections import Counter
from dataclasses import dataclass
from itertools import chain
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

# 深度學習與 HuggingFace：用中文 BERT 產生句向量與 token contextual embedding。
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoModel, AutoTokenizer
except ImportError as exc:
    raise ImportError("請先安裝 torch 與 transformers：%pip install torch transformers") from exc

# UMAP / Plotly / widgets 是視覺化與人工標註用；缺少時主流程仍可部分執行。
try:
    import umap.umap_ as umap
except ImportError:
    umap = None
    warnings.warn("缺少 umap-learn；UMAP 視覺化 cell 會略過。安裝：%pip install umap-learn")

try:
    import plotly.express as px
except ImportError:
    px = None
    warnings.warn("缺少 plotly；互動式 3D 圖會略過。安裝：%pip install plotly")

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    warnings.warn("缺少 ipywidgets；人工標註介面會略過。安裝：%pip install ipywidgets")

# 固定 random seed，讓 KMeans、PCA、抽樣與 MLP 訓練較容易重現。
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 專案資料與輸出位置集中定義，後續 cell 不再硬編路徑。
DATA_DIR = Path("Data")
KEYWORD_PATH = DATA_DIR / "negative_keywords.txt"
CSV_PATH = DATA_DIR / "nodes_page_1hop.csv"
OUTPUT_DIR = Path("outputs") / "concept_subspace"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 主要實驗參數集中在 CONFIG，調整 K、batch size、threshold 時只要改這裡。
CONFIG = {
    "model_name": "ckiplab/bert-base-chinese",  # 預設中文 BERT 模型，用於句向量與 token 向量
    "text_column_candidates": ["name", "group_name", "title"],
    "n_clusters": 30,  # Step 2 要拆成 30 個概念群
    "cluster_core_top_n": 5,
    "embedding_batch_size": 32,
    "max_length": 64,
    "umap_n_neighbors": 15,
    "umap_min_dist": 0.1,
    "subspace_k": 5,  # 每個 concept subspace 取前 k 個 PCA 主方向
    "max_risk_train_texts": 5000,
    "max_token_train_rows": 200000,
    "risk_epochs": 8,
    "risk_batch_size": 512,
    "risk_learning_rate": 1e-3,
    "final_threshold": 0.35,  # 最終融合分數超過此值才判定違規
    "token_contribution_threshold": 0.08,
}

# Cluster 人工標註選項；後續只有 MALICIOUS_LABELS 會被拿來建立惡意子空間。
LABEL_OPTIONS = ["無法判定", "黃", "賭", "詐", "金融違規", "非惡意內容"]
MALICIOUS_LABELS = ["黃", "賭", "詐", "金融違規"]
CONCEPT_DISPLAY_NAMES = {
    "黃": "色情內容",
    "賭": "賭博博弈",
    "詐": "金融詐騙",
    "金融違規": "金融違規",
}

# 這些詞只用於：自動初標、MLP pseudo token label、子空間 anchor fallback。
# 真正可信的概念來源仍以 Step 2 人工確認的 cluster label 為主。
# 概念詞典是 warm-start/弱監督工具，不是最終真值；最終仍以人工 cluster label 為準。
CONCEPT_KEYWORDS = {
    "黃": [
        "包養", "援交", "約炮", "外約", "茶莊", "魚訊", "小姐", "妹", "按摩", "半套", "全套",
        "八大", "夜場", "酒店", "招待所", "成人", "情色", "裸聊", "私密", "陪酒", "傳播妹",
    ],
    "賭": [
        "娛樂城", "博弈", "運彩", "賭", "下注", "百家樂", "棋牌", "六合彩", "彩票", "球版",
        "德州", "賽馬", "真人視訊", "老虎機", "返水", "賠率", "莊家",
    ],
    "詐": [
        "倍投", "帶單", "飆股", "內線", "保本", "穩賺", "致富", "槓桿", "投資群", "老師帶",
        "代操", "日領", "兼職", "不問經歷", "高薪", "借款", "貸款", "信貸", "快易貸", "企業貸",
        "二胎", "卡債", "強力過件", "免頭期", "法拍", "債務", "破產", "清算", "協商",
    ],
    "金融違規": [
        "貸款", "信貸", "快易貸", "企業貸", "借款", "二胎", "卡債", "強力過件", "免頭期",
        "債務", "破產", "法拍", "清算", "協商", "保本", "槓桿", "倍投", "內線", "帶單",
    ],
}

# 常見誤報提示詞，例如「大酒店」通常是住宿場域，不一定是色情語境。
BENIGN_HINTS = ["大酒店", "飯店", "旅館", "住宿", "餐廳", "精品店", "精品咖啡", "精品旅館", "精品家具"]
EPS = 1e-8

print(f"輸出目錄：{OUTPUT_DIR.resolve()}")
print(f"Embedding 模型：{CONFIG['model_name']}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 89.6 MB/s eta 0:00:00
輸出目錄：/content/outputs/concept_subspace
Embedding 模型：ckiplab/bert-base-chinese


## Step 1: Seed-based Corpus Filtering

讀取 seed keywords 與社團名稱資料，篩出包含任一 seed 的候選文本。候選集會保存為 `outputs/concept_subspace/step1_seed_candidates.csv`。


In [2]:
# 讀取人工 seed keywords；這些詞只負責初步召回候選樣本。
def read_seed_keywords(path: Path) -> List[str]:
    words = []
    with path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            word = line.strip()
            if word:
                words.append(word)
    return sorted(set(words), key=len, reverse=True)


# 自動尋找社團名稱欄位，避免資料欄位名稱稍有不同時整份 notebook 失效。
def find_text_column(df: pd.DataFrame, candidates: Sequence[str]) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"找不到文本欄位；目前欄位：{list(df.columns)}")


# 回傳某筆文本命中的所有 seed；保留清單是為了後續解釋為何被篩入候選集。
def find_seed_hits(text: str, seed_keywords: Sequence[str]) -> List[str]:
    return [kw for kw in seed_keywords if kw and kw in text]


def save_json_list_column(df: pd.DataFrame, col: str) -> pd.Series:
    return df[col].map(lambda xs: json.dumps(list(xs), ensure_ascii=False))


seed_keywords = read_seed_keywords(KEYWORD_PATH)
# nodes_page_1hop.csv 可能很大；全部用字串讀取可避免 page_id 或空值被 pandas 自動轉型。
raw_df = pd.read_csv(CSV_PATH, encoding="utf-8-sig", dtype=str, low_memory=False)
text_col = find_text_column(raw_df, CONFIG["text_column_candidates"])

# 清掉空白與缺失值，確保後續 seed match / embedding 只處理有效名稱。
raw_df[text_col] = raw_df[text_col].fillna("").astype(str).str.strip()
raw_df = raw_df[raw_df[text_col].str.len() > 0].reset_index(drop=True)
raw_df["matched_seed_keywords"] = raw_df[text_col].map(lambda t: find_seed_hits(t, seed_keywords))

# Step 1 的候選定義：只要命中任一 seed 就先召回；是否真惡意留到 Step 2 標註。
candidates = raw_df[raw_df["matched_seed_keywords"].map(bool)].copy().reset_index(drop=True)
candidates["candidate_text"] = candidates[text_col]
candidates["embedding_index"] = np.arange(len(candidates))

step1_path = OUTPUT_DIR / "step1_seed_candidates.csv"
# list 欄位寫入 CSV 前轉成 JSON 字串，避免 Excel/CSV 將清單格式弄壞。
step1_save = candidates.copy()
step1_save["matched_seed_keywords"] = save_json_list_column(step1_save, "matched_seed_keywords")
step1_save.to_csv(step1_path, index=False, encoding="utf-8-sig")

print(f"Seed keywords：{len(seed_keywords)} 個")
print(f"原始有效文本：{len(raw_df):,}")
print(f"候選惡意文本：{len(candidates):,}")
print(f"Step 1 output：{step1_path}")
display(candidates[[text_col, "matched_seed_keywords"]].head(10))


Seed keywords：30 個
原始有效文本：2,030,756
候選惡意文本：17,503
Step 1 output：outputs/concept_subspace/step1_seed_candidates.csv


,name,matched_seed_keywords
0,Est.citys時尚流行精品,[精品]
1,小資女精品服飾,[精品]
2,Alice 愛麗絲精品,[精品]
3,Ula日式精品,[精品]
4,Yuan精品服飾,[精品]
5,Ms Nana正韓精品服飾,[精品]
6,紗法亞精品婚紗 Sapphire Wedding,[精品]
7,Lillian Store 精品代購網,[精品]
8,米米亞精品童裝,[精品]
9,Judy-Korea-韓國精品服飾,[精品]


In [3]:
# 優先使用 GPU；沒有 CUDA 時自動退回 CPU。
def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 載入 tokenizer 與 BERT encoder；encoder.eval() 表示只做特徵抽取，不更新 BERT 權重。
def load_encoder(model_name: str = CONFIG["model_name"]) -> Tuple[Any, Any, torch.device]:
    device = get_device()
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    if not getattr(tokenizer, "is_fast", False):
        warnings.warn("目前 tokenizer 不是 fast tokenizer；token offset 可能無法用於 attention rollback。")
    return tokenizer, model, device


# 把每個 token 的 hidden state 平均成句向量；special tokens 會被排除。
def mean_pool_hidden(
    last_hidden: torch.Tensor,
    attention_mask: torch.Tensor,
    special_tokens_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    mask = attention_mask.float()
    if special_tokens_mask is not None:
        mask = mask * (1.0 - special_tokens_mask.float())
    mask = mask.unsqueeze(-1)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=EPS)
    return summed / denom


def embed_texts(
    texts: Sequence[str],
    tokenizer: Any,
    model: Any,
    device: torch.device,
    batch_size: int = CONFIG["embedding_batch_size"],
    max_length: int = CONFIG["max_length"],
    cache_path: Optional[Path] = None,
) -> np.ndarray:
    texts = [str(t) for t in texts]
    # Embedding 計算昂貴，因此用 cache；同時檢查文本清單是否一致，避免錯用舊向量。
    if cache_path and cache_path.exists():
        cached = np.load(cache_path, allow_pickle=True)
        cached_texts = cached["texts"].tolist()
        if cached_texts == texts:
            print(f"載入 embedding cache：{cache_path}")
            return cached["embeddings"].astype(np.float32)
        print("Embedding cache 文本不一致，重新計算。")

    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        # tokenizer 會做截斷/補齊，並回傳 BERT 所需的 input_ids 與 attention_mask。
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
            return_special_tokens_mask=True,
        )
        # [CLS]、[SEP]、[PAD] 不代表原始社團名稱內容，所以 mean pooling 時排除。
        special_tokens_mask = encoded.pop("special_tokens_mask").to(device)
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.inference_mode():
            outputs = model(**encoded)
            # 這裡得到 Step 1/2 clustering 用的句向量 embedding。
            pooled = mean_pool_hidden(outputs.last_hidden_state, encoded["attention_mask"], special_tokens_mask)
        all_embeddings.append(pooled.detach().cpu().numpy().astype(np.float32))
        if (start // batch_size + 1) % 25 == 0:
            print(f"已完成 {min(start + batch_size, len(texts)):,}/{len(texts):,}")

    embeddings = np.vstack(all_embeddings) if all_embeddings else np.empty((0, model.config.hidden_size), dtype=np.float32)
    if cache_path:
        np.savez_compressed(cache_path, embeddings=embeddings, texts=np.array(texts, dtype=object))
        print(f"寫入 embedding cache：{cache_path}")
    return embeddings


def reduce_to_3d(embeddings: np.ndarray, random_state: int = SEED) -> np.ndarray:
    if len(embeddings) == 0:
        return np.empty((0, 3), dtype=np.float32)
    # UMAP 需要足夠樣本；樣本太少或套件缺失時用 PCA 當穩定 fallback。
    if len(embeddings) < 4 or umap is None:
        n_components = min(3, embeddings.shape[1], len(embeddings))
        coords = PCA(n_components=n_components, random_state=random_state).fit_transform(embeddings)
        if coords.shape[1] < 3:
            coords = np.pad(coords, ((0, 0), (0, 3 - coords.shape[1])))
        return coords.astype(np.float32)
    # UMAP 使用 cosine metric，較符合 embedding 空間中的語意相似度。
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=min(CONFIG["umap_n_neighbors"], max(2, len(embeddings) - 1)),
        min_dist=CONFIG["umap_min_dist"],
        metric="cosine",
        random_state=random_state,
    )
    return reducer.fit_transform(embeddings).astype(np.float32)


def plot_umap_3d(
    coords: np.ndarray,
    frame: pd.DataFrame,
    color_col: str,
    title: str,
    hover_cols: Optional[List[str]] = None,
):
    if px is None:
        print("缺少 plotly，略過繪圖。")
        return None
    plot_df = frame.copy().reset_index(drop=True)
    if color_col in plot_df.columns:
        plot_df[color_col] = plot_df[color_col].map(
            lambda v: json.dumps(list(v), ensure_ascii=False) if isinstance(v, (list, tuple, set)) else str(v)
        )
    plot_df["umap_x"] = coords[:, 0]
    plot_df["umap_y"] = coords[:, 1]
    plot_df["umap_z"] = coords[:, 2]
    hover_cols = hover_cols or ["candidate_text", "matched_seed_keywords"]
    fig = px.scatter_3d(
        plot_df,
        x="umap_x",
        y="umap_y",
        z="umap_z",
        color=color_col,
        hover_data=[c for c in hover_cols if c in plot_df.columns],
        title=title,
        opacity=0.78,
        height=760,
    )
    fig.update_traces(marker={"size": 4})
    fig.show()
    return fig


# 下載/載入 ckiplab/bert-base-chinese，第一次執行可能需要較久。
# 第一次執行會從 HuggingFace 下載模型；之後會使用本機 cache。
tokenizer, encoder, device = load_encoder(CONFIG["model_name"])

embedding_cache = OUTPUT_DIR / "step1_candidate_embeddings_ckiplab_bert_base_chinese.npz"
# 將候選社團名稱轉成句向量，這些向量會被 Step 2 KMeans 使用。
candidate_embeddings = embed_texts(
    candidates["candidate_text"].tolist(),
    tokenizer=tokenizer,
    model=encoder,
    device=device,
    cache_path=embedding_cache,
)

step1_umap = reduce_to_3d(candidate_embeddings)
np.save(OUTPUT_DIR / "step1_umap_3d.npy", step1_umap)
plot_umap_3d(step1_umap, candidates, color_col="matched_seed_keywords", title="Step 1 Seed Candidate Embeddings - UMAP 3D")
print(candidate_embeddings.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstr

已完成 800/17,503
已完成 1,600/17,503
已完成 2,400/17,503
已完成 3,200/17,503
已完成 4,000/17,503
已完成 4,800/17,503
已完成 5,600/17,503
已完成 6,400/17,503
已完成 7,200/17,503
已完成 8,000/17,503
已完成 8,800/17,503
已完成 9,600/17,503
已完成 10,400/17,503
已完成 11,200/17,503
已完成 12,000/17,503
已完成 12,800/17,503
已完成 13,600/17,503
已完成 14,400/17,503
已完成 15,200/17,503
已完成 16,000/17,503
已完成 16,800/17,503
寫入 embedding cache：outputs/concept_subspace/step1_candidate_embeddings_ckiplab_bert_base_chinese.npz


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(17503, 768)


## Step 2: Embedding Space Clustering & Labeling

將候選文本 embedding 用 K-Means 分成 30 群，列出每群離中心最近的 5 筆。Notebook 會先給一個規則式初標，之後可用 `ipywidgets` 人工修正。標註結果會保存為：

- `step2_cluster_core_texts.csv`
- `step2_cluster_labels.csv`
- `step2_candidate_cluster_labels.csv`


In [4]:
# 用 KMeans 將候選樣本切成語意概念群；每個群之後由人判定概念屬性。
def run_kmeans(embeddings: np.ndarray, n_clusters: int = CONFIG["n_clusters"]) -> Tuple[np.ndarray, np.ndarray, KMeans]:
    if len(embeddings) == 0:
        raise ValueError("沒有候選 embedding 可聚類。")
    # 若候選數少於 30，K 不能超過樣本數。
    actual_k = min(n_clusters, len(embeddings))
    km = KMeans(n_clusters=actual_k, random_state=SEED, n_init=20)
    labels = km.fit_predict(embeddings)
    # 距離 cluster center 越近，越能代表該群核心語意。
    distances = np.linalg.norm(embeddings - km.cluster_centers_[labels], axis=1)
    return labels, distances, km


# 統計每群最常出現的 seed，輔助理解 cluster 的主題。
def top_seed_terms(rows: pd.DataFrame, top_n: int = 8) -> List[str]:
    counts = Counter(chain.from_iterable(rows["matched_seed_keywords"].tolist()))
    return [term for term, _ in counts.most_common(top_n)]


# 規則式初標只用來減少人工標註成本；請在 widget 或 CSV 中人工校正。
def rule_based_label_cluster(core_texts: Sequence[str], seed_terms: Sequence[str]) -> str:
    joined = " ".join(map(str, core_texts)) + " " + " ".join(seed_terms)
    # 先處理明顯 benign 的誤報，例如「大酒店」通常不是色情內容。
    if any(hint in joined for hint in BENIGN_HINTS) and not any(
        x in joined for x in ["外約", "包養", "博弈", "投注", "信貸", "強力過件"]
    ):
        return "非惡意內容"
    scores = {}
    for label, keywords in CONCEPT_KEYWORDS.items():
        scores[label] = sum(joined.count(kw) for kw in keywords)
    best_label, best_score = max(scores.items(), key=lambda kv: kv[1])
    if best_score <= 0:
        return "無法判定"
    return best_label


def make_cluster_summary(frame: pd.DataFrame, core_top_n: int = CONFIG["cluster_core_top_n"]) -> pd.DataFrame:
    rows = []
    for cluster_id, group in frame.groupby("cluster_id"):
        # 每群取離中心最近的 5 筆，讓人標註時看最具代表性的文本。
        core = group.nsmallest(core_top_n, "distance_to_cluster_center")
        core_texts = core["candidate_text"].tolist()
        seeds = top_seed_terms(group)
        auto_label = rule_based_label_cluster(core_texts, seeds)
        rows.append(
            {
                "cluster_id": int(cluster_id),
                "size": int(len(group)),
                "top_seed_keywords": json.dumps(seeds, ensure_ascii=False),
                "core_texts": json.dumps(core_texts, ensure_ascii=False),
                "auto_label": auto_label,
                "human_label": auto_label,
                "notes": "",
            }
        )
    return pd.DataFrame(rows).sort_values(["cluster_id"]).reset_index(drop=True)


cluster_labels, cluster_distances, kmeans_model = run_kmeans(candidate_embeddings)
candidates["cluster_id"] = cluster_labels
candidates["distance_to_cluster_center"] = cluster_distances

cluster_summary = make_cluster_summary(candidates)
cluster_core_path = OUTPUT_DIR / "step2_cluster_core_texts.csv"
cluster_summary.to_csv(cluster_core_path, index=False, encoding="utf-8-sig")

print(f"K-Means clusters：{cluster_summary.shape[0]}")
print(f"Cluster core texts output：{cluster_core_path}")
display(cluster_summary)


K-Means clusters：30
Cluster core texts output：outputs/concept_subspace/step2_cluster_core_texts.csv


,cluster_id,size,top_seed_keywords,core_texts,auto_label,human_label,notes
0,0,594,"[""精品""]","[""榮興精品（藝品 水晶 木頭）"", ""良慶閣精品佛具嘉義店"", ""天藝國際 木藝 沉香 茶...",無法判定,無法判定,
1,1,396,"[""精品""]","[""業興汽車精品百貨"", ""瑞夆汽車精品百貨"", ""LaBe 樂比精品百貨館"", ""薆妮爾 ...",無法判定,無法判定,
2,2,536,"[""精品""]","[""Vs精品服飾"", ""Clara精品服飾"", ""Fabulous 精品服飾"", ""Open...",無法判定,無法判定,
3,3,1183,"[""精品"", ""酒店"", ""致富"", ""厭世"", ""槓桿"", ""八大"", ""招待所"", ""借款""]","[""MINE MINE 韓國東大門精品服飾配件代購批發許願池"", ""CocoAi 日韓女...",黃,黃,
4,4,765,"[""借款"", ""貸款"", ""二胎"", ""全額"", ""法拍"", ""強力過件"", ""精品"", ""...","[""台中立榮當舖/汽機車借款/不限車齡/免留車/黃金珠寶典當質借"", ""宏福資產管理 全台...",詐,詐,
5,5,981,"[""精品"", ""破產"", ""致富""]","[""B'S SHOP 服飾精品"", ""Bibo'S Korea服飾精品"", ""️ LAURA...",詐,詐,
6,6,472,"[""娛樂城"", ""博弈""]","[""金利娛樂城"", ""Super8娛樂城"", ""Leo娛樂城"", ""GZ娛樂城"", ""Ddy...",賭,賭,
7,7,1056,"[""精品"", ""酒店"", ""強力過件"", ""八大""]","[""Pin品.歐巴精品賣場"", ""OMG 歐買嘎 時尚精品館"", ""風采華歌爾內衣精品店"",...",黃,黃,
8,8,291,"[""精品""]","[""眾讃國際精品"", ""艾夏Ishop國際精品"", ""沄豹國際精品"", ""DZ 國際精品"",...",無法判定,無法判定,
9,9,273,"[""精品""]","[""SUN精品"", ""ONON 精品"", ""X6 精品"", ""Shanghao精品"", ""B...",無法判定,無法判定,


In [5]:
# 建立 human-in-the-loop 標註介面：每個 cluster 一個下拉選單與備註欄。
def build_labeling_widget(cluster_summary: pd.DataFrame):
    if widgets is None:
        print("缺少 ipywidgets；請直接編輯 step2_cluster_core_texts.csv 的 human_label 欄位後，重新執行下一個 cell。")
        return None

    dropdowns = {}
    notes_boxes = {}
    rows = []
    for _, row in cluster_summary.iterrows():
        cid = int(row["cluster_id"])
        core_texts = json.loads(row["core_texts"])
        header = widgets.HTML(value=f"<b>Cluster {cid}</b> | size={row['size']} | seed={html.escape(row['top_seed_keywords'])}")
        core_html = widgets.HTML(value="<br>".join(f"{i+1}. {html.escape(t)}" for i, t in enumerate(core_texts)))
        # human_label 是後續建立概念子空間的依據，因此這裡的修正很重要。
        dropdown = widgets.Dropdown(options=LABEL_OPTIONS, value=row["human_label"], description="Label")
        notes = widgets.Textarea(value=str(row.get("notes", "")), description="Notes", layout=widgets.Layout(width="95%", height="45px"))
        dropdowns[cid] = dropdown
        notes_boxes[cid] = notes
        rows.append(widgets.VBox([header, core_html, dropdown, notes], layout=widgets.Layout(border="1px solid #ddd", padding="8px", margin="4px")))

    save_button = widgets.Button(description="Save labels", button_style="primary")
    output = widgets.Output()

    def on_save(_):
        out = cluster_summary.copy()
        out["human_label"] = out["cluster_id"].map(lambda cid: dropdowns[int(cid)].value)
        out["notes"] = out["cluster_id"].map(lambda cid: notes_boxes[int(cid)].value)
        label_path = OUTPUT_DIR / "step2_cluster_labels.csv"
        # 儲存後，下一個 cell 會讀取這份 CSV 並把 label 套回所有候選文本。
        out.to_csv(label_path, index=False, encoding="utf-8-sig")
        with output:
            output.clear_output()
            print(f"已儲存：{label_path}")

    save_button.on_click(on_save)
    display(widgets.VBox([widgets.HTML("<h3>Cluster Labeling</h3>"), *rows, save_button, output]))


build_labeling_widget(cluster_summary)

# 若不使用 widget，至少先保存自動初標，讓後續 cell 可以直接跑完。
label_path = OUTPUT_DIR / "step2_cluster_labels.csv"
if not label_path.exists():
    cluster_summary.to_csv(label_path, index=False, encoding="utf-8-sig")
    print(f"已寫入初始標註檔：{label_path}")


已寫入初始標註檔：outputs/concept_subspace/step2_cluster_labels.csv


In [6]:
# 優先讀取人工標註檔；若尚未標註，先用 auto_label 讓流程能完整跑通。
def load_cluster_labels(path: Path, fallback: pd.DataFrame) -> pd.DataFrame:
    if path.exists():
        labels = pd.read_csv(path, encoding="utf-8-sig")
    else:
        labels = fallback.copy()
    labels["cluster_id"] = labels["cluster_id"].astype(int)
    labels["human_label"] = labels["human_label"].where(labels["human_label"].isin(LABEL_OPTIONS), labels["auto_label"])
    return labels


cluster_labels_df = load_cluster_labels(OUTPUT_DIR / "step2_cluster_labels.csv", cluster_summary)
cluster_to_label = dict(zip(cluster_labels_df["cluster_id"].astype(int), cluster_labels_df["human_label"]))
cluster_to_auto = dict(zip(cluster_labels_df["cluster_id"].astype(int), cluster_labels_df["auto_label"]))

candidates_labeled = candidates.copy()
# 將 cluster-level label 展開到每一筆文本，形成後續 MLP 與 subspace 的訓練來源。
candidates_labeled["concept_label"] = candidates_labeled["cluster_id"].map(cluster_to_label).fillna("無法判定")
candidates_labeled["auto_label"] = candidates_labeled["cluster_id"].map(cluster_to_auto).fillna("無法判定")

step2_candidate_path = OUTPUT_DIR / "step2_candidate_cluster_labels.csv"
step2_save = candidates_labeled.copy()
step2_save["matched_seed_keywords"] = save_json_list_column(step2_save, "matched_seed_keywords")
step2_save.to_csv(step2_candidate_path, index=False, encoding="utf-8-sig")

step2_umap = reduce_to_3d(candidate_embeddings)
np.save(OUTPUT_DIR / "step2_umap_3d.npy", step2_umap)
# 以人工/初始標籤上色，檢查同概念是否在 embedding 空間中形成相近區域。
plot_umap_3d(
    step2_umap,
    candidates_labeled,
    color_col="concept_label",
    title="Step 2 Cluster Labels - UMAP 3D",
    hover_cols=["candidate_text", "cluster_id", "concept_label", "matched_seed_keywords"],
)

print(f"Step 2 candidate labels output：{step2_candidate_path}")
display(candidates_labeled["concept_label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count"))


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Step 2 candidate labels output：outputs/concept_subspace/step2_candidate_cluster_labels.csv


,label,count
0,黃,5783
1,詐,5181
2,無法判定,3552
3,非惡意內容,2145
4,賭,842


## Step 3: Contextual Co-occurrence & Attention MLP

這個 MLP **不直接做最終分類**。它吃 BERT contextual token embedding 與句向量互動特徵，輸出每個 token 的 `0~1` 語境風險權重。由於目前沒有 token-level 標籤，本 notebook 使用 Step 2 的 cluster label 建立弱監督 pseudo label：

- 人工標為 `黃 / 賭 / 詐 / 金融違規` 的文本中，命中概念詞或 seed 的 token 給高風險。
- 惡意文本中的其他 token 給較低但非零的語境背景風險。
- `非惡意內容` 給 0。
- `無法判定` 預設不進入訓練。


In [7]:
# 將關鍵詞轉成原始文字中的 char span，之後用 token offset 對齊到 BERT token。
def find_spans(text: str, terms: Sequence[str]) -> List[Tuple[int, int, str]]:
    spans = []
    for term in sorted(set(terms), key=len, reverse=True):
        if not term:
            continue
        for match in re.finditer(re.escape(term), text):
            spans.append((match.start(), match.end(), term))
    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    return spans


# 判斷某個 token 的字元範圍是否碰到任一關鍵詞 span。
def overlaps(offset: Tuple[int, int], spans: Sequence[Tuple[int, int, str]]) -> bool:
    start, end = offset
    if end <= start:
        return False
    return any(start < span_end and end > span_start for span_start, span_end, _ in spans)


def clean_token(token: str) -> str:
    return token.replace("##", "")


# 取得 token-level contextual embeddings；這是 Step 3 風險權重與 Step 5 回溯的核心資料。
def encode_token_records(
    texts: Sequence[str],
    tokenizer: Any,
    model: Any,
    device: torch.device,
    max_length: int = CONFIG["max_length"],
    batch_size: int = CONFIG["embedding_batch_size"],
) -> List[Dict[str, Any]]:
    if not getattr(tokenizer, "is_fast", False):
        raise ValueError("需要 fast tokenizer 才能取得 offset_mapping 以做 attention rollback。")

    records = []
    for start in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[start : start + batch_size]]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
            return_offsets_mapping=True,
            return_special_tokens_mask=True,
        )
        # offset_mapping 讓 token 可以回指原始社團名稱中的字元位置。
        offsets = encoded.pop("offset_mapping").cpu().numpy()
        special = encoded.pop("special_tokens_mask").cpu().numpy()
        input_ids = encoded["input_ids"].cpu().numpy()
        attention = encoded["attention_mask"].cpu().numpy()
        model_inputs = {k: v.to(device) for k, v in encoded.items()}
        with torch.inference_mode():
            # last_hidden_state 保留每個 token 在整句語境下的 contextual embedding。
            hidden = model(**model_inputs).last_hidden_state.detach().cpu().numpy().astype(np.float32)

        for i, text in enumerate(batch):
            token_rows = []
            for j in range(input_ids.shape[1]):
                if attention[i, j] == 0 or special[i, j] == 1:
                    continue
                off = tuple(map(int, offsets[i, j]))
                if off[1] <= off[0]:
                    continue
                token = tokenizer.convert_ids_to_tokens(int(input_ids[i, j]))
                token_rows.append({"token": token, "clean_token": clean_token(token), "offset": off, "embedding": hidden[i, j]})
            # 句向量作為語境摘要，與 token 向量一起餵給 MLP 學習共現風險。
            sent_embedding = np.mean([r["embedding"] for r in token_rows], axis=0).astype(np.float32) if token_rows else np.zeros(model.config.hidden_size, dtype=np.float32)
            records.append({"text": text, "tokens": token_rows, "sentence_embedding": sent_embedding})
    return records


# MLP 的特徵包含 token、句向量、乘積與差值，用來捕捉「詞在此語境中」是否危險。
def make_interaction_feature(token_emb: np.ndarray, sentence_emb: np.ndarray) -> np.ndarray:
    return np.concatenate([token_emb, sentence_emb, token_emb * sentence_emb, np.abs(token_emb - sentence_emb)]).astype(np.float32)


# 輕量 MLP：輸入 token-context 特徵，輸出單一 logit，經 sigmoid 後為 0~1 風險權重。
class ContextRiskMLP(nn.Module):
    def __init__(self, hidden_size: int, dropout: float = 0.15):
        super().__init__()
        input_dim = hidden_size * 4
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


# 用 cluster label 產生弱監督 token labels；這不是最終分類器，只學語境共現風險。
def build_token_training_arrays(
    frame: pd.DataFrame,
    tokenizer: Any,
    model: Any,
    device: torch.device,
    max_texts: int = CONFIG["max_risk_train_texts"],
    max_rows: int = CONFIG["max_token_train_rows"],
) -> Tuple[np.ndarray, np.ndarray]:
    trainable = frame[frame["concept_label"].isin(MALICIOUS_LABELS + ["非惡意內容"])].copy()
    if len(trainable) > max_texts:
        trainable = trainable.sample(max_texts, random_state=SEED)

    texts = trainable["candidate_text"].tolist()
    labels = trainable["concept_label"].tolist()
    records = encode_token_records(texts, tokenizer, model, device)

    X_rows, y_rows = [], []
    for rec, label in zip(records, labels):
        text = rec["text"]
        sentence_emb = rec["sentence_embedding"]
        malicious = label in MALICIOUS_LABELS
        terms = list(seed_keywords)
        if label in CONCEPT_KEYWORDS:
            terms += CONCEPT_KEYWORDS[label]
        spans = find_spans(text, terms)

        for token_row in rec["tokens"]:
            token_emb = token_row["embedding"]
            X_rows.append(make_interaction_feature(token_emb, sentence_emb))
            if malicious:
                # 惡意文本中命中 seed/概念詞的 token 給 1.0，其他語境 token 給 0.30。
                y_rows.append(1.0 if overlaps(token_row["offset"], spans) else 0.30)
            else:
                # 非惡意 cluster 的 token 視為低風險。
                y_rows.append(0.0)

    if not X_rows:
        raise ValueError("沒有可訓練的 token rows；請先完成人工標註或確認候選資料。")

    X = np.vstack(X_rows).astype(np.float32)
    y = np.array(y_rows, dtype=np.float32)
    if len(X) > max_rows:
        idx = np.random.default_rng(SEED).choice(len(X), size=max_rows, replace=False)
        X, y = X[idx], y[idx]
    return X, y


# 訓練 MLP 讓它學會 token 與整句語境的風險關係。
def train_context_risk_mlp(
    X: np.ndarray,
    y: np.ndarray,
    hidden_size: int,
    device: torch.device,
    epochs: int = CONFIG["risk_epochs"],
    batch_size: int = CONFIG["risk_batch_size"],
    lr: float = CONFIG["risk_learning_rate"],
) -> Tuple[ContextRiskMLP, List[Dict[str, float]]]:
    stratify = (y > 0.5) if len(np.unique(y > 0.5)) > 1 else None
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=stratify)
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_x = torch.from_numpy(X_val).to(device)
    val_y = torch.from_numpy(y_val).to(device)

    model = ContextRiskMLP(hidden_size).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        for bx, by in loader:
            bx = bx.to(device)
            by = by.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(bx)
            loss = loss_fn(logits, by)
            loss.backward()
            opt.step()
            total_loss += float(loss.detach().cpu()) * len(bx)

        model.eval()
        with torch.inference_mode():
            val_logits = model(val_x)
            val_loss = float(loss_fn(val_logits, val_y).detach().cpu())
            # sigmoid 後才是 0~1 的 contextual risk weight。
            val_pred = torch.sigmoid(val_logits)
            val_mae = float(torch.mean(torch.abs(val_pred - val_y)).detach().cpu())
        row = {"epoch": epoch, "train_loss": total_loss / len(train_ds), "val_loss": val_loss, "val_mae": val_mae}
        history.append(row)
        print(row)
    return model, history


# 推論單句時，回傳每個 token 的 0~1 語境風險分數。
def predict_context_risk_for_record(record: Dict[str, Any], risk_model: ContextRiskMLP, device: torch.device) -> np.ndarray:
    if not record["tokens"]:
        return np.array([], dtype=np.float32)
    features = np.vstack([make_interaction_feature(t["embedding"], record["sentence_embedding"]) for t in record["tokens"]]).astype(np.float32)
    risk_model.eval()
    with torch.inference_mode():
        logits = risk_model(torch.from_numpy(features).to(device))
        scores = torch.sigmoid(logits).detach().cpu().numpy().astype(np.float32)
    return scores


In [8]:
# 將 Step 2 標註後的文本展開成 token-level 訓練資料。
X_token, y_token = build_token_training_arrays(candidates_labeled, tokenizer, encoder, device)
print(f"Token training matrix：X={X_token.shape}, y positive ratio={(y_token > 0.5).mean():.3f}")

# 只訓練輕量 MLP；BERT encoder 維持 frozen，避免小資料下過度微調。
risk_model, risk_history = train_context_risk_mlp(X_token, y_token, encoder.config.hidden_size, device)

risk_model_path = OUTPUT_DIR / "step3_context_risk_mlp.pt"
# 保存模型權重與訓練設定，之後可直接載入做 inference。
torch.save(
    {
        "model_name": CONFIG["model_name"],
        "hidden_size": encoder.config.hidden_size,
        "state_dict": risk_model.state_dict(),
        "config": CONFIG,
        "history": risk_history,
    },
    risk_model_path,
)

pd.DataFrame(risk_history).to_csv(OUTPUT_DIR / "step3_context_risk_mlp_history.csv", index=False, encoding="utf-8-sig")
print(f"Step 3 MLP output：{risk_model_path}")


Token training matrix：X=(54431, 3072), y positive ratio=0.175
{'epoch': 1, 'train_loss': 0.5561729187152834, 'val_loss': 0.5029153227806091, 'val_mae': 0.08166714757680893}
{'epoch': 2, 'train_loss': 0.47598272015013354, 'val_loss': 0.46866485476493835, 'val_mae': 0.051480669528245926}
{'epoch': 3, 'train_loss': 0.45916101795759345, 'val_loss': 0.4564760625362396, 'val_mae': 0.04496868699789047}
{'epoch': 4, 'train_loss': 0.45035086891218595, 'val_loss': 0.44837579131126404, 'val_mae': 0.031802523881196976}
{'epoch': 5, 'train_loss': 0.44438715512607313, 'val_loss': 0.4479493200778961, 'val_mae': 0.04197404906153679}
{'epoch': 6, 'train_loss': 0.44351782654425576, 'val_loss': 0.44861751794815063, 'val_mae': 0.03477666527032852}
{'epoch': 7, 'train_loss': 0.43666953180301404, 'val_loss': 0.4477309286594391, 'val_mae': 0.026766732335090637}
{'epoch': 8, 'train_loss': 0.4334936669966036, 'val_loss': 0.4426000416278839, 'val_mae': 0.022395439445972443}
Step 3 MLP output：outputs/concept_sub

## Step 4: Subspace Definition & Disambiguation

每個惡意概念的子空間由以下方式定義：

1. 從 Step 2 人工確認為 `黃 / 賭 / 詐 / 金融違規` 的群集文本中收集 anchor terms。
2. 對 anchor terms 取 embedding，計算 centroid。
3. 對 centered anchor embedding 做 PCA，取前 `k` 個主方向作為該概念的 orthonormal basis。
4. token embedding 投影到該子空間，計算 projection norm、angle cosine 與 centroid cosine，作為歧義詞在特定語境下偏向某概念的程度。


In [9]:
# 每個 ConceptSubspace 保存：概念名稱、anchor terms、質心、PCA basis 與 normalization scale。
@dataclass
class ConceptSubspace:
    label: str
    display_name: str
    terms: List[str]
    centroid: np.ndarray
    basis: np.ndarray
    scale: float
    anchor_projection_norms: List[float]


# 投影前先做 L2 normalize，讓 norm/ cosine 比較不受原始向量長度影響。
def l2_normalize_matrix(x: np.ndarray) -> np.ndarray:
    return normalize(x.astype(np.float32), norm="l2")


# QR 分解把 PCA directions 正交化，確保 basis 可用於標準投影。
def orthonormalize_rows(vectors: np.ndarray) -> np.ndarray:
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.ndim == 1:
        vectors = vectors.reshape(1, -1)
    q, _ = np.linalg.qr(vectors.T)
    return q.T.astype(np.float32)


# 投影矩陣 P = B^T B；B 的 row 是 orthonormal basis。
def projection_matrix_from_basis(basis: np.ndarray) -> np.ndarray:
    basis = orthonormalize_rows(basis)
    return basis.T @ basis


# 計算 token 向量落在特定惡意子空間上的強度與方向一致性。
def projection_features(x: np.ndarray, subspace: ConceptSubspace) -> Dict[str, float]:
    x = np.asarray(x, dtype=np.float32)
    centered = x - subspace.centroid
    # coeff 是 token 在每個子空間主方向上的座標。
    coeff = subspace.basis @ centered
    # projected 是 token 被投影回原 embedding 空間後的向量。
    projected = subspace.basis.T @ coeff
    projection_norm = float(np.linalg.norm(projected))
    centered_norm = float(np.linalg.norm(centered))
    # angle_cosine 越高，代表 token 方向越貼近該概念子空間。
    angle_cosine = projection_norm / (centered_norm + EPS)
    centroid_cosine = float(cosine_similarity(x.reshape(1, -1), subspace.centroid.reshape(1, -1))[0, 0])
    normalized_projection = min(projection_norm / (subspace.scale + EPS), 1.0)
    subspace_score = normalized_projection * max(angle_cosine, 0.0) * max(centroid_cosine, 0.0)
    return {
        "projection_norm": projection_norm,
        "normalized_projection": float(normalized_projection),
        "angle_cosine": float(angle_cosine),
        "centroid_cosine": centroid_cosine,
        "subspace_score": float(subspace_score),
    }


# 從人工標為特定概念的 cluster 中收集 anchor terms，作為子空間建構材料。
def collect_anchor_terms(frame: pd.DataFrame, label: str, min_terms: int = 3, max_terms: int = 80) -> List[str]:
    sub = frame[frame["concept_label"] == label]
    terms = []
    label_terms = CONCEPT_KEYWORDS.get(label, [])
    searchable_terms = sorted(set(seed_keywords + label_terms), key=len, reverse=True)
    for text in sub["candidate_text"].astype(str).tolist():
        for term in searchable_terms:
            if term in text:
                terms.append(term)
    counts = Counter(terms)
    ordered = [term for term, _ in counts.most_common(max_terms)]

    # 若人工標註資料太少，使用概念詞典作 warm-start fallback；後續應回到 Step 2 補標。
    if len(ordered) < min_terms:
        # 標註資料不足時只加入該概念自己的詞典，避免不同概念子空間互相污染。
        ordered = list(dict.fromkeys(ordered + label_terms))[:max_terms]
        warnings.warn(f"{label} 的人工 anchor terms 少於 {min_terms}，已加入概念詞典 fallback。")
    return ordered[:max_terms]


# 對 anchor embeddings 做 PCA，前 k 個主成分就是該惡意概念的子空間方向。
def build_concept_subspace(label: str, terms: List[str], term_embeddings: np.ndarray, k: int = CONFIG["subspace_k"]) -> ConceptSubspace:
    if len(terms) == 0:
        raise ValueError(f"{label} 沒有 anchor terms。")
    X = l2_normalize_matrix(term_embeddings)
    centroid = X.mean(axis=0).astype(np.float32)
    centered = X - centroid

    if len(terms) >= 2 and np.linalg.norm(centered) > EPS:
        n_components = min(k, len(terms) - 1, X.shape[1])
        pca = PCA(n_components=n_components, random_state=SEED)
        # PCA 在 centroid-centered 的 anchor vectors 上執行，捕捉概念內主要變化方向。
        pca.fit(centered)
        basis = orthonormalize_rows(pca.components_)
    else:
        basis = orthonormalize_rows(centroid)

    temp = ConceptSubspace(
        label=label,
        display_name=CONCEPT_DISPLAY_NAMES.get(label, label),
        terms=terms,
        centroid=centroid,
        basis=basis,
        scale=1.0,
        anchor_projection_norms=[],
    )
    raw_norms = []
    for x in X:
        centered_x = x - centroid
        raw_norms.append(float(np.linalg.norm(basis.T @ (basis @ centered_x))))
    # 用 anchor 投影長度的第 90 百分位當 scale，將 projection norm 壓到可比較範圍。
    temp.scale = max(float(np.percentile(raw_norms, 90)), EPS)
    temp.anchor_projection_norms = raw_norms
    return temp


# 分別為 黃/賭/詐/金融違規 建立獨立子空間，避免把不同惡意概念混成單一方向。
def build_all_subspaces(labels_frame: pd.DataFrame) -> Dict[str, ConceptSubspace]:
    subspaces = {}
    for label in MALICIOUS_LABELS:
        terms = collect_anchor_terms(labels_frame, label)
        term_cache = OUTPUT_DIR / f"step4_anchor_embeddings_{label}.npz"
        term_embeddings = embed_texts(terms, tokenizer, encoder, device, cache_path=term_cache)
        subspaces[label] = build_concept_subspace(label, terms, term_embeddings)
        print(f"{label} / {subspaces[label].display_name}: {len(terms)} anchors, basis={subspaces[label].basis.shape}, scale={subspaces[label].scale:.4f}")
    return subspaces


concept_subspaces = build_all_subspaces(candidates_labeled)

subspace_path = OUTPUT_DIR / "step4_concept_subspaces.pkl"
with subspace_path.open("wb") as f:
    pickle.dump(concept_subspaces, f)

anchor_table = []
for label, subspace in concept_subspaces.items():
    for term in subspace.terms:
        anchor_table.append({"label": label, "display_name": subspace.display_name, "term": term})
pd.DataFrame(anchor_table).to_csv(OUTPUT_DIR / "step4_anchor_terms.csv", index=False, encoding="utf-8-sig")
print(f"Step 4 subspace output：{subspace_path}")


寫入 embedding cache：outputs/concept_subspace/step4_anchor_embeddings_黃.npz
黃 / 色情內容: 29 anchors, basis=(5, 768), scale=0.7231
寫入 embedding cache：outputs/concept_subspace/step4_anchor_embeddings_賭.npz
賭 / 賭博博弈: 16 anchors, basis=(5, 768), scale=0.7279
寫入 embedding cache：outputs/concept_subspace/step4_anchor_embeddings_詐.npz
詐 / 金融詐騙: 27 anchors, basis=(5, 768), scale=0.6897
寫入 embedding cache：outputs/concept_subspace/step4_anchor_embeddings_金融違規.npz
金融違規 / 金融違規: 19 anchors, basis=(5, 768), scale=0.6818
Step 4 subspace output：outputs/concept_subspace/step4_concept_subspaces.pkl


/tmp/ipykernel_4014/393342585.py:74: UserWarning:

金融違規 的人工 anchor terms 少於 3，已加入概念詞典 fallback。



## Step 5: Pipeline Integration & Attention Rollback

`evaluate_group_name(text)` 會對輸入社團名稱執行：

1. 取得 contextual token embeddings。
2. 用 Step 3 MLP 產生每個 token 的語境風險權重。
3. 對每個惡意 concept subspace 計算 projection features。
4. 融合分數：`context_risk_weight * subspace_score`。
5. 若最佳概念分數超過 threshold，輸出惡意判定與貢獻詞。


In [10]:
# 將 token-level 分數合併回可讀詞彙；優先合併已知 anchor/seed span，再補單 token。
def aggregate_contributing_words(
    text: str,
    token_rows: List[Dict[str, Any]],
    token_scores: np.ndarray,
    concept_label: str,
    top_n: int = 8,
    min_score: float = CONFIG["token_contribution_threshold"],
) -> List[Tuple[str, float]]:
    if len(token_rows) == 0:
        return []
    terms = []
    if concept_label in concept_subspaces:
        terms += concept_subspaces[concept_label].terms
    terms += CONCEPT_KEYWORDS.get(concept_label, []) + seed_keywords
    # 先找出原文中可形成完整詞的 span，例如「帶單」不要拆成「帶」「單」。
    spans = find_spans(text, terms)

    used = np.zeros(len(token_rows), dtype=bool)
    candidates_out = []
    for span_start, span_end, term in spans:
        idxs = [i for i, row in enumerate(token_rows) if row["offset"][0] < span_end and row["offset"][1] > span_start]
        if not idxs:
            continue
        score = float(np.max(token_scores[idxs]))
        if score >= min_score:
            candidates_out.append((term, score))
            used[idxs] = True

    for i, row in enumerate(token_rows):
        if used[i]:
            continue
        score = float(token_scores[i])
        if score >= min_score:
            start, end = row["offset"]
            word = text[start:end] or row["clean_token"]
            candidates_out.append((word, score))

    merged = {}
    for word, score in candidates_out:
        merged[word] = max(float(score), merged.get(word, 0.0))
    return sorted(merged.items(), key=lambda kv: kv[1], reverse=True)[:top_n]


# 最終 inference 函數：輸入社團名稱，輸出判定、命中概念與可解釋貢獻詞。
def evaluate_group_name(text: str, threshold: float = CONFIG["final_threshold"], top_n: int = 8) -> Dict[str, Any]:
    # 先拿到每個 token 的 contextual embedding 與原文字元 offset。
    records = encode_token_records([text], tokenizer, encoder, device, batch_size=1)
    record = records[0]
    token_rows = record["tokens"]
    if not token_rows:
        return {
            "text": text,
            "is_malicious": False,
            "matched_concept": "非惡意內容",
            "risk_score": 0.0,
            "concept_scores": {},
            "contributing_words": [],
            "token_details": [],
        }

    # Step 3 輸出：每個 token 在目前語境下的風險權重。
    context_risk = predict_context_risk_for_record(record, risk_model, device)
    token_embeddings = np.vstack([row["embedding"] for row in token_rows]).astype(np.float32)
    # Step 4 投影使用 normalize 後的 token 向量，和 anchor subspace 定義保持一致。
    token_embeddings = l2_normalize_matrix(token_embeddings)

    concept_token_scores = {}
    concept_projection_details = {}
    concept_scores = {}

    for label, subspace in concept_subspaces.items():
        per_token = []
        projection_rows = []
        for token_emb, risk in zip(token_embeddings, context_risk):
            # 計算 token 對目前概念子空間的 projection norm / angle / centroid similarity。
            proj = projection_features(token_emb, subspace)
            # Score Fusion：語境風險權重 x 子空間投影分數。
            fused = float(risk) * proj["subspace_score"]
            per_token.append(fused)
            projection_rows.append(proj)
        per_token = np.array(per_token, dtype=np.float32)
        concept_token_scores[label] = per_token
        concept_projection_details[label] = projection_rows
        # 用最高的少數 token 代表整個社團名稱風險，避免長文本平均稀釋關鍵詞。
        top_scores = np.sort(per_token)[-min(3, len(per_token)) :]
        concept_scores[label] = float(np.mean(top_scores)) if len(top_scores) else 0.0

    best_label = max(concept_scores, key=concept_scores.get)
    best_score = concept_scores[best_label]
    # Thresholding：最佳概念分數超過門檻才判定違規。
    is_malicious = bool(best_score >= threshold)
    matched_concept = CONCEPT_DISPLAY_NAMES.get(best_label, best_label) if is_malicious else "非惡意內容"
    contributing_words = aggregate_contributing_words(text, token_rows, concept_token_scores[best_label], best_label, top_n=top_n)

    # token_details 保留每個 token 的風險與投影細節，供 attention rollback 與 debug 使用。
    token_details = []
    for i, row in enumerate(token_rows):
        token_details.append(
            {
                "token": row["token"],
                "text_span": text[row["offset"][0] : row["offset"][1]],
                "offset": row["offset"],
                "context_risk": float(context_risk[i]),
                "best_concept_score": float(concept_token_scores[best_label][i]),
                "best_projection": concept_projection_details[best_label][i],
            }
        )

    return {
        "text": text,
        "is_malicious": is_malicious,
        "matched_concept": matched_concept,
        "risk_score": float(best_score),
        "concept_scores": {CONCEPT_DISPLAY_NAMES.get(k, k): float(v) for k, v in concept_scores.items()},
        "contributing_words": [(word, float(round(score, 4))) for word, score in contributing_words],
        "token_details": token_details,
    }


# 將 token 分數回填到原始字元，用紅色深淺顯示哪些字貢獻了判定。
def render_attention_rollback(result: Dict[str, Any]) -> HTML:
    text = result["text"]
    char_scores = np.zeros(len(text), dtype=np.float32)
    for row in result.get("token_details", []):
        start, end = row["offset"]
        # 多個 token/詞重疊時取最大分數，讓高貢獻字元更清楚。
        char_scores[start:end] = np.maximum(char_scores[start:end], row["best_concept_score"])
    max_score = float(char_scores.max()) if len(char_scores) else 0.0

    pieces = []
    for i, ch in enumerate(text):
        alpha = 0.0 if max_score <= EPS else min(float(char_scores[i] / max_score), 1.0)
        alpha = 0.10 + 0.75 * alpha if alpha > 0 else 0.0
        pieces.append(f'<span style="background: rgba(220, 53, 69, {alpha:.2f}); padding:2px 1px; border-radius:2px;">{html.escape(ch)}</span>')

    concept_scores = "<br>".join(f"{html.escape(k)}: {v:.4f}" for k, v in result["concept_scores"].items())
    words = ", ".join(f"{html.escape(w)}({s:.2f})" for w, s in result["contributing_words"])
    body = f'''
    <div style="font-family: system-ui, sans-serif; line-height:1.8;">
      <div><b>is_malicious:</b> {result["is_malicious"]}</div>
      <div><b>matched_concept:</b> {html.escape(result["matched_concept"])}</div>
      <div><b>risk_score:</b> {result["risk_score"]:.4f}</div>
      <div><b>contributing_words:</b> {words}</div>
      <div style="margin:8px 0; font-size:20px;">{"".join(pieces)}</div>
      <details><summary>concept scores</summary>{concept_scores}</details>
    </div>
    '''
    return HTML(body)


# Demo：可替換為任意社團名稱。
examples = [
    "老師帶單飆股保本投資群",
    "娛樂城百家樂返水群",
    "圓山大酒店住宿優惠",
    "日領高薪不問經歷兼職",
]
for sample in examples:
    result = evaluate_group_name(sample)
    display(render_attention_rollback(result))
    print(json.dumps({k: v for k, v in result.items() if k != "token_details"}, ensure_ascii=False, indent=2))


{
  "text": "老師帶單飆股保本投資群",
  "is_malicious": false,
  "matched_concept": "非惡意內容",
  "risk_score": 0.03562542423605919,
  "concept_scores": {
    "色情內容": 0.03562542423605919,
    "賭博博弈": 0.0305465217679739,
    "金融詐騙": 0.03133171796798706,
    "金融違規": 0.0316026508808136
  },
  "contributing_words": []
}


{
  "text": "娛樂城百家樂返水群",
  "is_malicious": false,
  "matched_concept": "非惡意內容",
  "risk_score": 0.23551888763904572,
  "concept_scores": {
    "色情內容": 0.1901179403066635,
    "賭博博弈": 0.23551888763904572,
    "金融詐騙": 0.194369837641716,
    "金融違規": 0.06376946717500687
  },
  "contributing_words": [
    [
      "娛樂城",
      0.2638
    ],
    [
      "百家樂",
      0.2211
    ]
  ]
}


{
  "text": "圓山大酒店住宿優惠",
  "is_malicious": false,
  "matched_concept": "非惡意內容",
  "risk_score": 0.18537412583827972,
  "concept_scores": {
    "色情內容": 0.16247105598449707,
    "賭博博弈": 0.07120818644762039,
    "金融詐騙": 0.18537412583827972,
    "金融違規": 0.04287049546837807
  },
  "contributing_words": [
    [
      "酒店",
      0.2606
    ]
  ]
}


{
  "text": "日領高薪不問經歷兼職",
  "is_malicious": false,
  "matched_concept": "非惡意內容",
  "risk_score": 0.03928576782345772,
  "concept_scores": {
    "色情內容": 0.03928576782345772,
    "賭博博弈": 0.02995772659778595,
    "金融詐騙": 0.03681941702961922,
    "金融違規": 0.024301692843437195
  },
  "contributing_words": []
}


## 主要輸出檔案

- `step1_seed_candidates.csv`：seed 篩選後的候選文本。
- `step1_candidate_embeddings_ckiplab_bert_base_chinese.npz`：候選文本 embedding。
- `step1_umap_3d.npy` / `step2_umap_3d.npy`：UMAP 3D 座標。
- `step2_cluster_core_texts.csv`：每個 cluster 的核心文本與初標。
- `step2_cluster_labels.csv`：人工/初始 cluster 標註。
- `step2_candidate_cluster_labels.csv`：候選文本、cluster 與標籤。
- `step3_context_risk_mlp.pt`：語境風險 MLP。
- `step4_anchor_terms.csv`：概念子空間 anchor terms。
- `step4_concept_subspaces.pkl`：概念子空間物件。

最終推論函數：

```python
evaluate_group_name("老師帶單飆股保本投資群")
```

回傳格式包含：`is_malicious`, `matched_concept`, `contributing_words`, `risk_score`, `concept_scores`, `token_details`。
